In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import tree
import optuna
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [15]:
def print_classification_metrics(y_true, y_pred):
    print(f'Accuracy: {accuracy_score(y_true, y_pred)}')
    print(classification_report(y_true, y_pred))

In [16]:
data = pd.read_csv('data/k4_eda.csv')
y = data['Fire Alarm']
X = data.drop(columns=['Fire Alarm', 'UTC'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### DecisionTreeClassifier

In [17]:
reg = DecisionTreeClassifier(max_depth=6, criterion='gini', random_state=42)
reg.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=6, random_state=42)

In [18]:
y_pred = reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

Accuracy: 0.9990943476639497
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      5343
        True       1.00      1.00      1.00     13428

    accuracy                           1.00     18771
   macro avg       1.00      1.00      1.00     18771
weighted avg       1.00      1.00      1.00     18771



#### Подбор гиперпараметров

In [19]:
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 1, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        criterion='gini',
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2025-06-06 08:09:16,294] A new study created in memory with name: no-name-3153df7f-9afd-44f3-8361-10a607f79e36
[I 2025-06-06 08:09:16,506] Trial 0 finished with value: 0.9910722539269458 and parameters: {'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 11, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9910722539269458.
[I 2025-06-06 08:09:16,736] Trial 1 finished with value: 0.9932185577157169 and parameters: {'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 18, 'max_features': 'log2'}. Best is trial 1 with value: 0.9932185577157169.
[I 2025-06-06 08:09:17,186] Trial 2 finished with value: 0.994520042268331 and parameters: {'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 2 with value: 0.994520042268331.
[I 2025-06-06 08:09:17,454] Trial 3 finished with value: 0.9946341870931452 and parameters: {'max_depth': 19, 'min_samples_split': 14, 'min_samples_leaf': 13, 'max_features': 'sqrt'}. Best is trial 

In [20]:
reg = DecisionTreeClassifier(**study.best_params, criterion='gini', random_state=42)
reg.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=24, min_samples_split=8, random_state=42)

In [21]:
y_pred = reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

Accuracy: 0.9994139896649087
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      5343
        True       1.00      1.00      1.00     13428

    accuracy                           1.00     18771
   macro avg       1.00      1.00      1.00     18771
weighted avg       1.00      1.00      1.00     18771



### BaggingClassifier

In [22]:
base_classifier = DecisionTreeClassifier(max_depth=6, criterion='gini', random_state=42)
bg_reg = BaggingClassifier(
    estimator=base_classifier,
    n_estimators=100,
    random_state=42
)
bg_reg.fit(X_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=6,
                                                   random_state=42),
                  n_estimators=100, random_state=42)

In [23]:
y_pred = bg_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

Accuracy: 0.9990943476639497
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      5343
        True       1.00      1.00      1.00     13428

    accuracy                           1.00     18771
   macro avg       1.00      1.00      1.00     18771
weighted avg       1.00      1.00      1.00     18771



#### Подбор гиперпараметров

In [24]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_samples = trial.suggest_float('max_samples', 0.1, 1.0)
    max_features = trial.suggest_float('max_features', 0.1, 1.0)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(criterion='gini', random_state=42),
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2025-06-06 08:09:45,573] A new study created in memory with name: no-name-d7cd1135-2432-4907-a183-b756b6be58dc
[I 2025-06-06 08:10:13,573] Trial 0 finished with value: 0.9996803314112398 and parameters: {'n_estimators': 146, 'max_samples': 0.6726584284560536, 'max_features': 0.4333973890082341, 'bootstrap': True}. Best is trial 0 with value: 0.9996803314112398.
[I 2025-06-06 08:10:22,161] Trial 1 finished with value: 0.9992921592454674 and parameters: {'n_estimators': 135, 'max_samples': 0.14818725245177972, 'max_features': 0.49139923544226993, 'bootstrap': False}. Best is trial 0 with value: 0.9996803314112398.
[I 2025-06-06 08:10:30,856] Trial 2 finished with value: 0.9993834964792899 and parameters: {'n_estimators': 47, 'max_samples': 0.405053599174074, 'max_features': 0.9840665191795952, 'bootstrap': True}. Best is trial 0 with value: 0.9996803314112398.
[I 2025-06-06 08:10:55,593] Trial 3 finished with value: 0.9995889993905811 and parameters: {'n_estimators': 126, 'max_samples

In [25]:
bg_reg = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion='gini', random_state=42),
    **study.best_params,
    random_state=42
)
bg_reg.fit(X_train, y_train)

BaggingClassifier(bootstrap=False,
                  estimator=DecisionTreeClassifier(random_state=42),
                  max_features=0.5171752748165908,
                  max_samples=0.8392484939216962, n_estimators=114,
                  random_state=42)

In [26]:
y_pred = bg_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

Accuracy: 0.9997869053326941
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      5343
        True       1.00      1.00      1.00     13428

    accuracy                           1.00     18771
   macro avg       1.00      1.00      1.00     18771
weighted avg       1.00      1.00      1.00     18771



### GradientBoostingClassifier

In [27]:
gb_reg = GradientBoostingClassifier(max_depth=6, n_estimators=100, random_state=42)
gb_reg.fit(X_train, y_train)

GradientBoostingClassifier(max_depth=6, random_state=42)

In [28]:
y_pred = gb_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

Accuracy: 0.9995738106653881
              precision    recall  f1-score   support

       False       1.00      1.00      1.00      5343
        True       1.00      1.00      1.00     13428

    accuracy                           1.00     18771
   macro avg       1.00      1.00      1.00     18771
weighted avg       1.00      1.00      1.00     18771



#### Подбор гиперпараметров

In [29]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    max_depth = trial.suggest_int('max_depth', 1, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        subsample=subsample,
        random_state=42
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2025-06-06 08:36:17,587] A new study created in memory with name: no-name-2a5b7cb7-84e7-4637-86f5-56b29f206eef
[I 2025-06-06 08:36:56,078] Trial 0 finished with value: 0.9993606628224798 and parameters: {'n_estimators': 172, 'learning_rate': 0.09951266398163612, 'max_depth': 3, 'min_samples_split': 11, 'min_samples_leaf': 14, 'subsample': 0.6621941565456491}. Best is trial 0 with value: 0.9993606628224798.
[I 2025-06-06 08:37:39,368] Trial 1 finished with value: 0.9992921644586312 and parameters: {'n_estimators': 133, 'learning_rate': 0.031062782943770073, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 19, 'subsample': 0.6035394228808366}. Best is trial 0 with value: 0.9993606628224798.
[I 2025-06-06 08:38:37,497] Trial 2 finished with value: 0.9996575029675935 and parameters: {'n_estimators': 119, 'learning_rate': 0.18767080088350255, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 10, 'subsample': 0.6540604913690559}. Best is trial 2 with value: 0.9996575

KeyboardInterrupt: 

In [ ]:
gb_reg = GradientBoostingClassifier(**study.best_params, random_state=42)
gb_reg.fit(X_train, y_train)

In [ ]:
y_pred = gb_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

### StackingClassifier

In [ ]:
estimators = [
    ('dt', DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)),
    ('lr', LogisticRegression(random_state=42))
]
stack_reg = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(random_state=42))
stack_reg.fit(X_train, y_train)

In [ ]:
y_pred = stack_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    dt_max_depth = trial.suggest_int('dt_max_depth', 1, 10)
    dt_min_samples_split = trial.suggest_int('dt_min_samples_split', 2, 20)
    C = trial.suggest_float('C', 0.1, 10.0)
    estimators = [
        ('dt', DecisionTreeClassifier(max_depth=dt_max_depth, min_samples_split=dt_min_samples_split, criterion='gini', random_state=42)),
        ('lr', LogisticRegression(random_state=42))
    ]
    meta_model = LogisticRegression(C=C, random_state=42)
    model = StackingClassifier(estimators=estimators, final_estimator=meta_model)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
best_params = study.best_params
best_estimators = [
    ('dt', DecisionTreeClassifier(max_depth=best_params['dt_max_depth'], min_samples_split=best_params['dt_min_samples_split'], criterion='gini', random_state=42)),
    ('lr', LogisticRegression(random_state=42))
]
best_meta_model = LogisticRegression(C=best_params['C'], random_state=42)
stack_reg = StackingClassifier(estimators=best_estimators, final_estimator=best_meta_model)
stack_reg.fit(X_train, y_train)

In [ ]:
y_pred = stack_reg.predict(X_test)

print_classification_metrics(y_test, y_pred)

### CatBoostClassifier

In [ ]:
cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=3,
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train, y_train)

In [ ]:
y_pred = cat_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    iterations = trial.suggest_int('iterations', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    depth = trial.suggest_int('depth', 1, 10)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1, 10)
    bagging_temperature = trial.suggest_float('bagging_temperature', 0, 1)
    model = CatBoostClassifier(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        bagging_temperature=bagging_temperature,
        random_seed=42,
        verbose=0
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
cat_model = CatBoostClassifier(**study.best_params, random_seed=42, verbose=0)
cat_model.fit(X_train, y_train)

In [ ]:
y_pred = cat_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

### XGBClassifier

In [ ]:
xgb_model = XGBClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 1, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    gamma = trial.suggest_float('gamma', 0, 5)
    reg_alpha = trial.suggest_float('reg_alpha', 0, 10)
    reg_lambda = trial.suggest_float('reg_lambda', 0, 10)
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        verbosity=0
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
xgb_model = XGBClassifier(**study.best_params, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

### LGBMClassifier

In [ ]:
lgb_model = LGBMClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)

In [ ]:
y_pred = lgb_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 1, 10)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    num_leaves = trial.suggest_int('num_leaves', 2, 100)
    min_child_samples = trial.suggest_int('min_child_samples', 1, 20)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 0, 10)
    reg_lambda = trial.suggest_float('reg_lambda', 0, 10)
    model = LGBMClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,
        verbose=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

In [ ]:
lgb_model = LGBMClassifier(**study.best_params, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)

In [ ]:
y_pred = lgb_model.predict(X_test)

print_classification_metrics(y_test, y_pred)

Все модели

In [ ]:
models = [reg, bg_reg, gb_reg, stack_reg, cat_model, xgb_model, lgb_model]

In [ ]:
model_names = [
    'Decision Tree Classifier',
    'Bagging Classifier',
    'Gradient Boosting Classifier',
    'Stacking Classifier',
    'CatBoost Classifier',
    'XGBoost Classifier',
    'LightGBM Classifier'
]

In [ ]:
results = []
for model, name in zip(models, model_names):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='binary') 
    results.append({'Model': name, 'Accuracy': accuracy, 'F1': f1})

df = pd.DataFrame(results)
df = df.sort_values(by='Accuracy', ascending=False)
print("Таблица сравнения:")
print(df)

plt.figure(figsize=(12, 6))
plt.bar(df['Model'], df['Accuracy'], color='skyblue')
plt.xlabel('Модель')
plt.ylabel('Accuracy')
plt.title('Сравнение классификационных моделей по Accuracy')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(df['Accuracy']):
    plt.text(i, v + 0.01, f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
class_names = ['No Alarm', 'Alarm']
fig = plt.figure(figsize=(15, 10))
tree.plot_tree(reg, feature_names=X_train.columns, filled=True, class_names=class_names)
plt.show()